In [22]:
import os
import yaml
import json
import logging
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from sklearn.metrics import classification_report
from pyspark.sql.types import FloatType
from pyspark.sql.functions import udf, col


from utils.spark_session import get_spark_session

In [23]:
spark = get_spark_session(app_name="06-model-final")

In [24]:
encoded_path = os.path.join("..", "data", "train_test", "train_encoded.parquet")
config_path = os.path.join("..", "src", "config", "feature_config.yaml")
params_path = os.path.join("..", "models", "tuning_best_params_gbt.json")
features_selected_path = os.path.join("..", "src", "features", "selected", "features_selected.yaml")
model_output_path = os.path.join("..", "models", "final_gbt_model")

In [25]:
with open(config_path, 'r') as f:
    feature_config = yaml.safe_load(f)
target_col = [k for k, v in feature_config.items() if isinstance(v, dict) and v.get("target")][0]

with open(features_selected_path, 'r') as f:
    selected_yaml = yaml.safe_load(f)
feature_cols = selected_yaml.get("support_random_forest", [])

In [26]:
feature_cols

['real_amount_per_limit',
 'amount_pct_limit',
 'real_amount',
 'limit_factor_vs_tx',
 'pct_current_vs_total_session',
 'amount_cumulative',
 'time_since_test_start']

In [27]:
train_df = spark.read.parquet(encoded_path)
train_df.show(5)

+--------------------+----------------+
|            features|target_converted|
+--------------------+----------------+
|[22.16,3.07777777...|               0|
|[13.57,1.88472222...|               1|
|[16.11,2.2375E-4,...|               1|
|[13.56,1.88333333...|               0|
|[12.27,1.70416666...|               1|
+--------------------+----------------+
only showing top 5 rows



In [28]:
train_df = train_df.drop("index")
train_df.show(5)

+--------------------+----------------+
|            features|target_converted|
+--------------------+----------------+
|[22.16,3.07777777...|               0|
|[13.57,1.88472222...|               1|
|[16.11,2.2375E-4,...|               1|
|[13.56,1.88333333...|               0|
|[12.27,1.70416666...|               1|
+--------------------+----------------+
only showing top 5 rows



In [29]:
if target_col != "label":
    train_df = train_df.withColumnRenamed(target_col, "label")

In [30]:
with open(params_path, 'r') as f:
    best_params = json.load(f)

In [31]:
train_df.show(truncate=False)

+------------------------------------------------------------------------------------------------------------------+-----+
|features                                                                                                          |label|
+------------------------------------------------------------------------------------------------------------------+-----+
|[22.16,3.077777777777778E-4,3249.0974729241875,9.5,3.077777777777778E-4,0.0,22.16]                                |0    |
|[13.57,1.8847222222222223E-4,5305.821665438467,17.25,1.8847222222222223E-4,22.16,0.6123646209386282]              |1    |
|[16.11,2.2375E-4,4469.27374301676,22.0,2.2375E-4,35.730000000000004,0.45088161209068006]                          |1    |
|[13.56,1.8833333333333335E-4,5309.734513274336,23.0,1.8833333333333335E-4,51.84,0.26157407407407407]              |0    |
|[12.27,1.7041666666666667E-4,5867.9706601467,24.0,1.7041666666666667E-4,65.4,0.18761467889908254]                 |1    |
|[12.36,1.716666

In [32]:
gbt = GBTClassifier(
    labelCol="label",
    featuresCol="features",
    maxDepth=int(best_params["maxDepth"]),
    maxIter=int(best_params["maxIter"]),
    stepSize=float(best_params["stepSize"]),
    seed=96
)

final_model = gbt.fit(train_df.select("features", "label"))
final_model.write().overwrite().save(model_output_path)

print(f"Final GBT model trained and saved to: {model_output_path}")

Final GBT model trained and saved to: ..\models\final_gbt_model


In [33]:
predictions = final_model.transform(train_df.select("features", "label"))

evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
auc = evaluator.evaluate(predictions)
print(f"AUC on training set: {auc:.4f}")

extract_prob_1 = udf(lambda v: float(v[1]), FloatType())
predictions = predictions.withColumn("prob_1", extract_prob_1(col("probability")))

pred_pd = predictions.select("prediction", "label", "prob_1").toPandas()

report = classification_report(
    y_true=pred_pd["label"],
    y_pred=pred_pd["prediction"],
    digits=4
)
print("\nClassification Report (Training Set):")
print(report)


AUC on training set: 0.9177

Classification Report (Training Set):
              precision    recall  f1-score   support

           0     0.8965    0.9301    0.9130     86719
           1     0.7157    0.6209    0.6649     24561

    accuracy                         0.8619    111280
   macro avg     0.8061    0.7755    0.7890    111280
weighted avg     0.8566    0.8619    0.8583    111280



In [34]:
spark.stop()